In [1]:
fn = "/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/tif/mtb_slice_5375_s0.ome.tif"

In [2]:
from skimage import io

In [5]:
%%time
image = io.imread(fn)

CPU times: user 19min 8s, sys: 3min 19s, total: 22min 28s
Wall time: 13min 10s


In [5]:
import tifffile
import dask.array as da
import napari
import zarr
from ome_zarr.io import parse_url
from ome_zarr.writer import write_image, add_metadata
from dask.diagnostics import ProgressBar
from pathlib import Path

In [15]:
%%time
zarr_image = tifffile.imread(fn, aszarr=True)
dask_image = da.from_zarr(zarr_image)

CPU times: user 51.6 ms, sys: 62.6 ms, total: 114 ms
Wall time: 850 ms


In [16]:
dask_image

dask.array<from-zarr, shape=(11, 3, 41702, 58291), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>

In [17]:
dask_image = dask_image.transpose(1, 0, 2, 3,)  # lazy; no data copy
dask_image

dask.array<transpose, shape=(3, 11, 41702, 58291), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>

In [21]:
import zarr
from ome_zarr.io import parse_url
from ome_zarr.writer import write_image, add_metadata
from dask.diagnostics import ProgressBar
from pathlib import Path

out_zarr = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/zarr/slice_5375.zarr")

# v0.5 / Zarr v3 store
store = parse_url(out_zarr, mode="w").store
root = zarr.group(store=store)

# writes data + builds a 2x YX pyramid by default
with ProgressBar():  # optional live progress
    write_image(
        image=dask_image,
        group=root,
        axes="czyx",
        storage_options=dict(chunks=(1, 1, 2048, 2048)),  # (C,Z,Y,X)
    )

# optional: channel labels for nicer viewing in napari/viv
add_metadata(root, {"omero": {
    "channels": [
        {"label": "CF405"},
        {"label": "CF488"},
        {"label": "CF561"},
    ]
}})


[########################################] | 100% Completed | 88m 0ss


In [6]:
out_zarr = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/zarr/slice_5375.zarr")


In [7]:
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader
import napari


# read the image data
reader = Reader(parse_url(out_zarr))
# nodes may include images, labels etc
nodes = list(reader())
# first node will be the image pixel data
image_node = nodes[0]

dask_data = image_node.data

# # We can view this in napari
# # NB: image axes are CZYX: split channels by C axis=0


In [8]:
dask_data

[dask.array<from-zarr, shape=(3, 11, 41702, 58291), dtype=uint16, chunksize=(1, 1, 2048, 2048), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 20851, 29145), dtype=uint16, chunksize=(1, 1, 2048, 2048), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 10425, 14572), dtype=uint16, chunksize=(1, 1, 2048, 2048), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 5212, 7286), dtype=uint16, chunksize=(1, 1, 2048, 2048), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 2606, 3643), dtype=uint16, chunksize=(1, 1, 2048, 2048), chunktype=numpy.ndarray>]

In [9]:
viewer = napari.view_image(dask_data, channel_axis=0)

/tmp/ipykernel_2169301/1715230323.py:1: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(dask_data, channel_axis=0)


In [38]:
for layer in viewer.layers:
    layer.scale = (2.0, 0.1625, 0.1625)

In [43]:
dask_data[1]

dask.array<from-zarr, shape=(3, 11, 20851, 29145), dtype=uint16, chunksize=(1, 1, 2048, 2048), chunktype=numpy.ndarray>

In [44]:
from ome_zarr.writer import write_multiscales_metadata

# after write_image(... axes="czyx")
level_names = sorted(root.array_keys(), key=int)   # <-- not group_keys()

axes = [
    {"name": "c", "type": "channel"},
    {"name": "z", "type": "space", "unit": "micrometer"},
    {"name": "y", "type": "space", "unit": "micrometer"},
    {"name": "x", "type": "space", "unit": "micrometer"},
]

px_z, px_y, px_x = 2.0, 0.1625, 0.1625
datasets = []
for i, p in enumerate(level_names):
    datasets.append({
        "path": p,
        "coordinateTransformations": [
            {"type": "scale", "scale": [1.0, px_z, px_y*(2**i), px_x*(2**i)]},  # C Z Y X
            {"type": "translation", "translation": [0, 0, 0, 0]},
        ]
    })

write_multiscales_metadata(root, datasets=datasets, axes=axes)


### Increasing I/O speed

In [48]:
dask_image_ROI = dask_image[:,:,0:10000, 0:10000]

In [49]:
dask_image_ROI

dask.array<getitem, shape=(3, 11, 10000, 10000), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>

In [50]:
import zarr
from ome_zarr.io import parse_url
from ome_zarr.writer import write_image, add_metadata

# your dask image: (Z, Y, X, C)
dimg_czyx = dimg.transpose(3, 0, 1, 2).rechunk((1, 1, 1024, 1024))

from zarr.codecs import Blosc  # Zarr v3 (NGFF v0.5)
compressors = (Blosc(cname="lz4", clevel=1, shuffle=Blosc.SHUFFLE),)

path = "/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/zarr/slice_5375_fast_view.ome.zarr"

store = parse_url(path, mode="w").store
root = zarr.group(store=store)

with ProgressBar():  # optional live progress
    write_image(
        image=dask_image,
        group=root,
        axes="czyx",
        storage_options=dict(chunks=(1, 1, 512, 512),)#) compressors=compressors),
    )

# optional labels so you can toggle channels quickly
add_metadata(root, {"omero": {"channels": [
    {"label": "CF405"}, {"label": "CF488"}, {"label": "CF561"}
]}})


[#####################                   ] | 53% Completed | 2hr 59ms

IOStream.flush timed out


[############################            ] | 70% Completed | 3hr 51m

IOStream.flush timed out


[########################################] | 100% Completed | 4hr 58m


In [51]:
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader
import napari


# read the image data
reader = Reader(parse_url(path))
# nodes may include images, labels etc
nodes = list(reader())
# first node will be the image pixel data
image_node = nodes[0]

dask_data = image_node.data

# # We can view this in napari
# # NB: image axes are CZYX: split channels by C axis=0


In [52]:
dask_data

[dask.array<from-zarr, shape=(3, 11, 41702, 58291), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 20851, 29145), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 10425, 14572), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 5212, 7286), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 2606, 3643), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>]

In [53]:
viewer = napari.view_image(dask_data, channel_axis=0)

/tmp/ipykernel_1425401/1715230323.py:1: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(dask_data, channel_axis=0)


# Todo:

1. Increase speed of loading by reducing chunk size
2. Create rapid label layer
3. Refine I/O process

### convert to a pyramidal OME-TIFF for napari using bftools in command line
(godspee) dayn@9GRLVQ3:/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/tif$ bfconvert mtb_slice_5375_s0.ome.tif mtb_slice_5375_pyr.ome.tiff -bigtiff -pyramid-scale 2 -pyramid-resolutions 5 -tilex 512 -tiley 512 -compression LZW



### View in napari

In [6]:
import napari

In [7]:
viewer = napari.Viewer()
viewer.add_image(image)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291, 3) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


<Image layer 'image' at 0x7c21dc5e6750>

In [8]:
image.shape

(11, 41702, 58291, 3)

In [9]:
viewer.add_image(image, channel_axis=-1)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


[<Image layer 'Image' at 0x7c222aa83b50>,
 <Image layer 'Image [1]' at 0x7c21dcf02cd0>,
 <Image layer 'Image [2]' at 0x7c21dc285bd0>]

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (11, 41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 16384 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTU

# My solution

In [24]:
image.shape

(11, 41702, 58291, 3)

# Chatgpt guff below

In [10]:
import numpy as np
import zarr
from openslide import OpenSlide


In [ ]:
OpenSlide.

In [ ]:
#!/usr/bin/env python3
# Minimal TIFF (z,y,x,c) -> pyramidal OME-Zarr (multiscales)

from pathlib import Path
import numpy as np
import tifffile as tiff
import dask.array as da
from dask.array import coarsen
import zarr
from numcodecs import Blosc
from ome_zarr.writer import write_multiscales_metadata
from tqdm.auto import tqdm

# --------- inputs / knobs ----------
tif_path = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/tif/mtb_slice_5375_s0.ome.tif")
out_zarr = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/zarr/slice_5375.zarr")


# chunking (z,y,x,c)
CHUNK = (1, 2048, 2048, 3)
COMP  = Blosc(cname="zstd", clevel=5, shuffle=Blosc.BITSHUFFLE)

# stop pyramid when both y & x <= this
TARGET_MIN_YX = 1024

# from your metadata
px_um = dict(z=2.0, y=0.1625, x=0.1625)
channel_labels = ["CF405", "CF488", "CF561"]
# ------------------------------------

# open lazily
asz = tiff.imread(tif_path, aszarr=True)
src = da.from_zarr(asz)  # expects (z,y,x,c)
assert src.ndim == 4, f"expected (z,y,x,c), got {src.shape}"
src = src.rechunk(CHUNK)

# helper: pad Y/X up to next multiple of 'factor'
def pad_even_yx(a, factor=2):
    pad = [(0,0)] * a.ndim
    for ax in (1, 2):  # y, x
        rem = a.shape[ax] % factor
        if rem:
            pad[ax] = (0, factor - rem)
    # edge padding duplicates edge pixels (cheap + safe for overviews)
    return da.pad(a, pad, mode="edge")

# build pyramid (2x YX) using safe coarsen
levels = [src]
scales = [[px_um["z"], px_um["y"], px_um["x"], 1.0]]

arr = src
while max(arr.shape[1:3]) > TARGET_MIN_YX:
    arr_even = pad_even_yx(arr, factor=2)
    arr = coarsen(np.mean, arr_even, {1: 2, 2: 2}, trim_excess=False)
    arr = arr.rechunk(CHUNK)
    levels.append(arr)
    i = len(levels) - 1
    scales.append([px_um["z"], px_um["y"] * 2**i, px_um["x"] * 2**i, 1.0])



In [17]:
src

dask.array<rechunk-merge, shape=(11, 3, 41702, 58291), dtype=uint16, chunksize=(1, 3, 2048, 3), chunktype=numpy.ndarray>

In [19]:
# ---- Zarr v2/v3 compressor shim ----
try:
    # Zarr v3 path
    from zarr.codecs import Blosc as ZarrBlosc
    CREATE_KW = {
        "compressors": (ZarrBlosc(cname="zstd", clevel=5, shuffle=ZarrBlosc.BITSHUFFLE),),
    }
except Exception:
    # Fallback for Zarr v2
    from numcodecs import Blosc as NumBlosc
    print('here')
    CREATE_KW = {
        "compressor": NumBlosc(cname="zstd", clevel=5, shuffle=NumBlosc.BITSHUFFLE),
    }


here


In [21]:
# zarr group
root = zarr.open_group(str(out_zarr), mode="w")


# chunk offset helper (for tqdm block writes)
def chunk_offsets(chunks_tuple):
    offs = [0]
    s = 0
    for ch in chunks_tuple[:-1]:
        s += ch
        offs.append(s)
    return offs

# --- stream-write each level with tqdm (Zarr v3-safe) ---
for i, arr in enumerate(levels):
    create_fn = getattr(root, "create_array", None) or root.create_dataset  # v3 then v2
    ds = create_fn(
        str(i),
        shape=arr.shape,
        chunks=CHUNK,
        dtype=arr.dtype,
        overwrite=True,
        # **CREATE_KW,  # <- key change here
    )

    nz, ny, nx, nc = arr.numblocks
    total_blocks = nz * ny * nx * nc

    def _offs(ch):
        o, out = 0, [0]
        for s in ch[:-1]:
            o += s
            out.append(o)
        return out

    z_off = _offs(arr.chunks[0])
    y_off = _offs(arr.chunks[1])
    x_off = _offs(arr.chunks[2])
    c_off = _offs(arr.chunks[3])

    from tqdm.auto import tqdm
    with tqdm(total=total_blocks, desc=f"Writing level {i} {arr.shape}") as pbar:
        for bz in range(nz):
            for by in range(ny):
                for bx in range(nx):
                    for bc in range(nc):
                        block = arr.blocks[bz, by, bx, bc].compute()
                        z0, y0, x0, c0 = z_off[bz], y_off[by], x_off[bx], c_off[bc]
                        z1, y1, x1, c1 = z0 + block.shape[0], y0 + block.shape[1], x0 + block.shape[2], c0 + block.shape[3]
                        ds[z0:z1, y0:y1, x0:x1, c0:c1] = block
                        pbar.update(1)


# multiscales metadata
axes = [
    {"name": "z", "type": "space", "unit": "micrometer"},
    {"name": "y", "type": "space", "unit": "micrometer"},
    {"name": "x", "type": "space", "unit": "micrometer"},
    {"name": "c", "type": "channel"},
]
datasets = [{"path": str(i)} for i in range(len(levels))]
coord = [
    [
        {"type": "scale", "scale": s},
        {"type": "translation", "translation": [0, 0, 0, 0]},
    ]
    for s in scales
]
write_multiscales_metadata(root, datasets=datasets, axes=axes, coordinateTransformations=coord)

# optional: simple OMERO block with channel labels (keeps viewers happy)
root.attrs["omero"] = {
    "name": out_zarr.stem,
    "channels": [{"label": lbl} for lbl in channel_labels],
}

print(f"Done: {out_zarr} with {len(levels)} levels")


Writing level 0 (11, 3, 41702, 58291):   0%|          | 0/4488561 [00:00<?, ?it/s]

KeyboardInterrupt: 